# 01 - Data Loading
Load CMEMS oceanographic data and AIS fishing effort (2015-2024)

In [1]:
!pip install -q xarray netCDF4 pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.0 MB/s eta 0:00:00


In [ ]:
# !wget -q -0 - ipv4.icanhazip.com

In [ ]:
# ! streamlit run Full_ConvLSTM.py & npx localtunnel --port 8501

In [2]:
from google.colab import drive
import xarray as xr
import pandas as pd
import numpy as np

drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/fishing_project/"

Mounted at /content/drive


In [3]:
# West Philippine Sea bounding box
LAT_MIN, LAT_MAX = 10, 20
LON_MIN, LON_MAX = 114, 120
DATE_START = "2019-01-01"
DATE_END   = "2024-12-31"

In [4]:
# Load oceanographic data
print("Loading CMEMS Global Ocean Physics Reanalysis...")
physics_ds = xr.open_dataset(DATA_DIR + "cmems_mod_glo_phy_my_0.083deg_P1M-m_1777301394294.nc")
print(f"Variables: {list(physics_ds.data_vars)}")
print(f"Shape: {physics_ds.dims}")

print("\nLoading CMEMS Global Ocean Biogeochemistry Hindcast...")
bgc_ds = xr.open_dataset(DATA_DIR + "cmems_mod_glo_bgc_my_0.25deg_P1M-m_1777301388002.nc")
print(f"Variables: {list(bgc_ds.data_vars)}")
print(f"Shape: {bgc_ds.dims}")

Loading CMEMS Global Ocean Physics Reanalysis...
Variables: ['thetao', 'uo', 'vo', 'zos']
Shape: FrozenMappingWarningOnValuesAccess({'time': 120, 'depth': 5, 'latitude': 121, 'longitude': 72})

Loading CMEMS Global Ocean Biogeochemistry Hindcast...
Variables: ['chl', 'nppv']
Shape: FrozenMappingWarningOnValuesAccess({'time': 120, 'depth': 5, 'latitude': 41, 'longitude': 25})


In [6]:
# Load AIS fishing effort data — flat folder structure (all years in one dir)
import glob, os

AIS_ROOT = DATA_DIR + "ais_fishing/"
USE_COLS = ["date", "cell_ll_lat", "cell_ll_lon", "fishing_hours"]

def load_ais_filtered(ais_root, date_start, date_end,
                      lat_min, lat_max, lon_min, lon_max):
    start = pd.Timestamp(date_start)
    end   = pd.Timestamp(date_end)
    chunks = []

    # All CSVs are flat in one folder — no year subfolders
    all_files = sorted(glob.glob(os.path.join(ais_root, "*.csv")))
    print(f"Total CSVs found in folder: {len(all_files)}")

    for fp in all_files:
        fname = os.path.basename(fp)
        # Extract date from filename: fleet-monthly-csvs-10-v3-YYYY-MM-DD.csv
        try:
            file_date = pd.Timestamp(fname[-14:-4])
        except Exception:
            continue
        # Skip files outside date range without opening them
        if not (start <= file_date <= end):
            continue

        try:
            df = pd.read_csv(
                fp,
                usecols=USE_COLS,
                dtype={
                    "cell_ll_lat":   "float32",
                    "cell_ll_lon":   "float32",
                    "fishing_hours": "float32",
                }
            )
            # Filter to bbox immediately (drops ~98% of rows)
            mask = (
                (df["cell_ll_lat"] >= lat_min) & (df["cell_ll_lat"] <  lat_max) &
                (df["cell_ll_lon"] >= lon_min) & (df["cell_ll_lon"] <  lon_max)
            )
            filtered = df[mask]
            if not filtered.empty:
                chunks.append(filtered)
                print(f"  {fname[-14:-4]}: {len(filtered)} rows in bbox")
        except Exception as e:
            print(f"  Skipped {fname}: {e}")

    if not chunks:
        raise ValueError("No AIS data found for the given bbox / date range!")

    ais_df = pd.concat(chunks, ignore_index=True)
    ais_df["date"]       = pd.to_datetime(ais_df["date"])
    ais_df["year_month"] = ais_df["date"].dt.to_period("M")
    return ais_df


print("Loading AIS fishing effort CSVs (2019-2024, bbox-filtered)...")
ais_df = load_ais_filtered(
    AIS_ROOT,
    DATE_START, DATE_END,       # picks up "2019-01-01" and "2024-12-31" automatically
    LAT_MIN, LAT_MAX, LON_MIN, LON_MAX
)

print(f"\nAIS records in bbox : {len(ais_df):,}")
print(f"Date range          : {ais_df['date'].min().date()} to {ais_df['date'].max().date()}")
print(f"Unique months       : {ais_df['year_month'].nunique()}")

Loading AIS fishing effort CSVs (2019-2024, bbox-filtered)...
Total CSVs found in folder: 72
  2019-01-01: 116 rows in bbox
  2019-02-01: 24 rows in bbox
  2019-03-01: 42 rows in bbox
  2019-04-01: 57 rows in bbox
  2019-05-01: 165 rows in bbox
  2019-06-01: 39 rows in bbox
  2019-07-01: 7 rows in bbox
  2019-08-01: 18 rows in bbox
  2019-09-01: 142 rows in bbox
  2019-10-01: 87 rows in bbox
  2019-11-01: 64 rows in bbox
  2019-12-01: 166 rows in bbox
  2020-01-01: 88 rows in bbox
  2020-02-01: 70 rows in bbox
  2020-03-01: 57 rows in bbox
  2020-04-01: 44 rows in bbox
  2020-05-01: 42 rows in bbox
  2020-06-01: 264 rows in bbox
  2020-07-01: 111 rows in bbox
  2020-08-01: 117 rows in bbox
  2020-09-01: 92 rows in bbox
  2020-10-01: 104 rows in bbox
  2020-11-01: 66 rows in bbox
  2020-12-01: 136 rows in bbox
  2021-01-01: 308 rows in bbox
  2021-02-01: 24 rows in bbox
  2021-03-01: 49 rows in bbox
  2021-04-01: 41 rows in bbox
  2021-05-01: 192 rows in bbox
  2021-06-01: 654 rows in b

In [7]:
# Time range verification
print("\n=== Time Range Verification ===")
print(f"Physics data: {physics_ds.time.min().values} to {physics_ds.time.max().values}")
print(f"BGC data: {bgc_ds.time.min().values} to {bgc_ds.time.max().values}")
print(f"AIS data: {ais_df['date'].min()} to {ais_df['date'].max()}")


=== Time Range Verification ===
Physics data: 2015-01-01T00:00:00.000000000 to 2024-12-01T00:00:00.000000000
BGC data: 2015-01-01T00:00:00.000000000 to 2024-12-01T00:00:00.000000000
AIS data: 2019-01-01 00:00:00 to 2024-12-01 00:00:00


In [8]:
# Save summary
summary = {
    'physics_shape': dict(physics_ds.dims),
    'bgc_shape': dict(bgc_ds.dims),
    # 'ais_records': len(ais_df),
    'date_range': f"{DATE_START} to {DATE_END}"
}

import json
with open(DATA_DIR + 'data_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✔ Data loading complete")


✔ Data loading complete


/tmp/ipykernel_1797/3833258255.py:3: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'physics_shape': dict(physics_ds.dims),
/tmp/ipykernel_1797/3833258255.py:4: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'bgc_shape': dict(bgc_ds.dims),
